# Benchmark Results Visualization
Comparing **gemma-4-E4B-it**, **Qwen3.5-4B**, and **Qwen3.5-9B** on financial Q&A tasks.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 11

: 

In [ ]:
# ── Raw data ──────────────────────────────────────────────────────────────────
models = ['gemma-4-E4B-it', 'Qwen3.5-4B', 'Qwen3.5-9B']
colors = ['#e07b54', '#4e9af1', '#57b06e']

overall = {
    'gemma-4-E4B-it': {'90Q': 73.3, 'MT': 55.6, 'Overall': 64.4},
    'Qwen3.5-4B':     {'90Q': 88.9, 'MT': 82.2, 'Overall': 85.6},
    'Qwen3.5-9B':     {'90Q': 85.6, 'MT': 68.9, 'Overall': 77.2},
}

# Multi-turn accuracy by conversation length
turn_counts = [1, 2, 3, 4, 5, 6, 12]
mt_by_turns = {
    'gemma-4-E4B-it': [72.5, 75.0, 12.5, 33.3, 14.3,  0.0,   0.0],
    'Qwen3.5-4B':     [88.2, 91.7, 75.0, 66.7, 57.1, 60.0, 100.0],
    'Qwen3.5-9B':     [74.5,100.0, 37.5, 66.7, 57.1, 20.0,   0.0],
}

# Sample counts per turn bucket
sample_counts = [51, 12, 8, 6, 7, 5, 1]

## 1 · Overall Accuracy by Task Type

In [ ]:
categories = ['90-Q (Single-turn)', 'Multi-turn', 'Overall']
keys       = ['90Q', 'MT', 'Overall']

x    = np.arange(len(categories))
w    = 0.25

fig, ax = plt.subplots(figsize=(9, 5))

for i, (model, color) in enumerate(zip(models, colors)):
    vals = [overall[model][k] for k in keys]
    bars = ax.bar(x + i * w, vals, w, label=model, color=color, alpha=0.88, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x + w)
ax.set_xticklabels(categories)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Overall Accuracy by Task Type', fontweight='bold', pad=12)
ax.set_ylim(0, 108)
ax.legend(loc='upper left')
ax.spines[['top', 'right']].set_visible(False)
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('benchmark_overall.png', bbox_inches='tight')
plt.show()

## 2 · Multi-turn Accuracy vs. Conversation Length

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x_pos = list(range(len(turn_counts)))
x_labels = [str(t) for t in turn_counts]

for model, color in zip(models, colors):
    ax.plot(x_pos, mt_by_turns[model], marker='o', color=color,
            linewidth=2, markersize=7, label=model)

# Annotate sample sizes
for xi, n in zip(x_pos, sample_counts):
    ax.annotate(f'n={n}', xy=(xi, 2), ha='center', fontsize=8,
                color='gray', style='italic')

ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels)
ax.set_xlabel('Number of Turns')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Multi-turn Accuracy by Conversation Length', fontweight='bold', pad=12)
ax.set_ylim(-5, 115)
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('benchmark_multiturn_by_turns.png', bbox_inches='tight')
plt.show()

## 3 · Accuracy Drop: Single-turn → Multi-turn

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

y = np.arange(len(models))
h = 0.35

st_vals = [overall[m]['90Q'] for m in models]
mt_vals = [overall[m]['MT']  for m in models]

bars_st = ax.barh(y + h/2, st_vals, h, label='Single-turn (90-Q)', color='#4e9af1', alpha=0.85)
bars_mt = ax.barh(y - h/2, mt_vals, h, label='Multi-turn',         color='#e07b54', alpha=0.85)

for bar, v in zip(bars_st, st_vals):
    ax.text(v + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{v:.1f}%', va='center', fontsize=9)
for bar, v, st in zip(bars_mt, mt_vals, st_vals):
    drop = st - v
    ax.text(v + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{v:.1f}%  (↓{drop:.1f}pp)', va='center', fontsize=9)

ax.set_yticks(y)
ax.set_yticklabels(models)
ax.set_xlabel('Accuracy (%)')
ax.set_title('Single-turn vs. Multi-turn Accuracy', fontweight='bold', pad=12)
ax.set_xlim(0, 112)
ax.legend(loc='lower right')
ax.spines[['top', 'right']].set_visible(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('benchmark_st_vs_mt.png', bbox_inches='tight')
plt.show()

## 4 · Heatmap: Accuracy per Model × Turn Depth

In [ ]:
import matplotlib.colors as mcolors

data = np.array([mt_by_turns[m] for m in models])

fig, ax = plt.subplots(figsize=(10, 3.5))
im = ax.imshow(data, cmap='RdYlGn', vmin=0, vmax=100, aspect='auto')

ax.set_xticks(range(len(turn_counts)))
ax.set_xticklabels([f'turns={t}\n(n={n})' for t, n in zip(turn_counts, sample_counts)])
ax.set_yticks(range(len(models)))
ax.set_yticklabels(models)
ax.set_title('Multi-turn Accuracy Heatmap (%) by Turn Depth', fontweight='bold', pad=12)

for i in range(len(models)):
    for j in range(len(turn_counts)):
        val = data[i, j]
        text_color = 'black' if 30 < val < 80 else 'white'
        ax.text(j, i, f'{val:.0f}%', ha='center', va='center',
                fontsize=10, color=text_color, fontweight='bold')

plt.colorbar(im, ax=ax, label='Accuracy (%)', shrink=0.85)
plt.tight_layout()
plt.savefig('benchmark_heatmap.png', bbox_inches='tight')
plt.show()